# 混合数据集

![](./xiaodongguaAIGC_dataset.png)

In [1]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

data_name1 = 'xiaodongguaAIGC/alpaca_gpt4_data_zh' # ignore
data_name2 = 'vicgalle/alpaca-gpt4'
data_name3 = 'LooksJuicy/ruozhiba'
data_name4 = 'silk-road/alpaca-data-gpt4-chinese' # 基于google翻译 仅演示如何按照比例来切割数据集

dataset1 = load_dataset(data_name1)
dataset2 = load_dataset(data_name2)
dataset3 = load_dataset(data_name3)
dataset4 = load_dataset(data_name4)

/Users/denghang/Miniconda3/envs/llm/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/denghang/Miniconda3/envs/llm/lib/python3.9/site-packages/huggingface_hub/repocard.py:105: UserWarning: Repo card metadata block was not found. Setting CardData to empty.
  warnings.warn("Repo card metadata block was not found. Setting CardData to empty.")


# dataset1: 中文alpaca数据集

In [2]:
dataset1 = load_dataset(data_name1)
print(dataset1)

/Users/denghang/Miniconda3/envs/llm/lib/python3.9/site-packages/huggingface_hub/repocard.py:105: UserWarning: Repo card metadata block was not found. Setting CardData to empty.
  warnings.warn("Repo card metadata block was not found. Setting CardData to empty.")


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 48818
    })
})

# dataset2: 英文 alpaca 数据集

In [3]:
# process dataset2
print(dataset2)
dataset2 = dataset2.remove_columns([
    'text',
])
print(dataset2)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 52002
    })
})

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 52002
    })
})

# dataset3:  弱智吧QA

In [4]:
# process dataset3
print(dataset3)
def process_ruozhiba(examples):
    examples['input'] = ''
    return examples
dataset3 = dataset3.map(process_ruozhiba, num_proc=8)
print(dataset3)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 1496
    })
})

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'input'],
        num_rows: 1496
    })
})

# dataset4:  另一个alpaca中文数据

In [5]:
print(dataset4)
def process_fn(examples):
    examples['instruction']=examples['instruction_zh']
    examples['input']=examples['input_zh']
    examples['output']=examples['output_zh']
    return examples
dataset4 = dataset4.map(process_fn, num_proc=8, remove_columns = ["instruction_zh", "input_zh", 'output_zh'])
dataset4['train'] = dataset4['train'].shard(num_shards=10, index=0) # 只取10%数据
print(dataset4)

DatasetDict({
    train: Dataset({
        features: ['instruction_zh', 'input_zh', 'output_zh', 'instruction', 'input', 'output'],
        num_rows: 52049
    })
})

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5205
    })
})

In [6]:
# 规则过滤: 基于长度过滤
print(dataset4)
dataset4['train'] = dataset4['train'].filter(lambda example: len(example["instruction"]+
                                                                 example["input"]+
                                                                 example["output"])<512)
print(dataset4)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5205
    })
})

Filter:   0%|          | 0/5205 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 4670
    })
})

# 合并数据集

In [7]:

dataset = concatenate_datasets([dataset1['train'], dataset2['train'], dataset3['train']])
dataset = DatasetDict({'train': dataset})
dataset = dataset.shuffle(seed=42)
print(dataset)
# print(dataset['train'][:20])
for i in range(20):
    print('-'*100)
    print('instruction: ', dataset['train']['instruction'][i])
    print('input: ', dataset['train']['input'][i])
    print('output: ', dataset['train']['output'][i])

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 102316
    })
})

----------------------------------------------------------------------------------------------------

instruction:  Generate a short description about a fictional character - a space traveler.

input:

output:  Meet Luna, the intrepid space traveler, on an endless journey through the cosmos. With her trusty 
spacecraft and unwavering determination, Luna explores the far reaches of the galaxy, encountering new worlds and 
civilizations along the way. Her bravery, intelligence, and endless curiosity make her the ideal explorer, always 
ready to embark on the next adventure. Luna is a true pioneer, forging a path through the unknown, in search of new
discoveries and experiences.

...

output:  "The Vibrantly Unique Boutique"

----------------------------------------------------------------------------------------------------

instruction:  找一个强调手眼协调的篮球训练。

input:

output:  好的！我找到了以下这个篮球训练，它可以帮助您提升手眼协调能力：

名字：俯卧撑篮球滚动

拓展方法: 
将两个篮球放在地上，面朝下，将手放在篮球上，进行俯卧撑。在完整的俯卧撑动作完成后，将手上的篮球滚动到身体的另一侧，
然后用另一只手接住篮球，再做另一次俯卧撑。反复练习这一动作。

目的：这项训练强调对篮球的控制能力，并要求您在训练中保持身体平衡。您需要积极地调整身体姿势和手眼协调，才能顺利完成
这项训练。经过训练，您的肌肉控制能力和手眼协调能力将会得到大大提升。

# 保存

In [8]:
dataset.save_to_disk('./output/alpaca_en_zh_ruozhiba')

Saving the dataset (0/1 shards):   0%|          | 0/102316 [00:00<?, ? examples/s]